In [1]:
# Install kagglehub if not already installed
!pip install -q kagglehub

#Data Cleanup & Merging

##Data Loading

In [2]:
import kagglehub
import pandas as pd
from pathlib import Path


path = kagglehub.dataset_download("unsdsn/world-happiness")
print("Path to dataset files:", path)

RAW_DIR = Path(path)


FILES = {
    "2015": RAW_DIR / "2015.csv",
    "2016": RAW_DIR / "2016.csv",
    "2017": RAW_DIR / "2017.csv",
    "2018": RAW_DIR / "2018.csv",
    "2019": RAW_DIR / "2019.csv",
}

dfs = {}
for year, path in FILES.items():
    df = pd.read_csv(path)
    dfs[year] = df
    print(f"Loaded {year}: {df.shape}")

Using Colab cache for faster access to the 'world-happiness' dataset.
Path to dataset files: /kaggle/input/world-happiness
Loaded 2015: (158, 12)
Loaded 2016: (157, 13)
Loaded 2017: (155, 12)
Loaded 2018: (156, 9)
Loaded 2019: (156, 9)


In [3]:
dfs["2019"].head()

,Overall rank,Country or region,Score,GDP per capita,Social support,Healthy life expectancy,Freedom to make life choices,Generosity,Perceptions of corruption
0,1,Finland,7.769,1.340,1.587,0.986,0.596,0.153,0.393
1,2,Denmark,7.600,1.383,1.573,0.996,0.592,0.252,0.410
2,3,Norway,7.554,1.488,1.582,1.028,0.603,0.271,0.341
3,4,Iceland,7.494,1.380,1.624,1.026,0.591,0.354,0.118
4,5,Netherlands,7.488,1.396,1.522,0.999,0.557,0.322,0.298


##Data Mapping correctly

In [4]:
COLUMN_MAP = {
    # Country column (varies by year)
    "Country": "Country",
    "Country or region": "Country",
    "Country name": "Country",

    # Happiness Score
    "Happiness Score": "happiness_score",
    "Happiness.Score": "happiness_score",
    "Score": "happiness_score",

    # Happiness Rank (not used in final output, but normalize anyway)
    "Happiness Rank": "happiness_rank",
    "Happiness.Rank": "happiness_rank",
    "Overall rank": "happiness_rank",

    # GDP
    "Economy (GDP per Capita)": "gdp_per_capita",
    "Economy..GDP.per.Capita.": "gdp_per_capita",
    "GDP per capita": "gdp_per_capita",

    # Social Support
    "Family": "social_support",
    "Family.": "social_support",
    "Social support": "social_support",

    # Health / Life Expectancy
    "Health (Life Expectancy)": "healthy_life_expectancy",
    "Health..Life.Expectancy.": "healthy_life_expectancy",
    "Healthy life expectancy": "healthy_life_expectancy",

    # Freedom
    "Freedom": "freedom",
    "Freedom.": "freedom",
    "Freedom to make life choices": "freedom",

    # Generosity
    "Generosity": "generosity",
    "Generosity.": "generosity",

    # Corruption
    "Trust (Government Corruption)": "corruption_perception",
    "Trust..Government.Corruption.": "corruption_perception",
    "Perceptions of corruption": "corruption_perception",
}

def standardize_columns(df):
    df = df.rename(columns=COLUMN_MAP)
    df.columns = [c.lower().replace(" ", "_") for c in df.columns]
    return df

##Country and Region Mapping using UN geoscheme region mapping

In [5]:
# Country name aliases (for inconsistent naming across years)
COUNTRY_ALIASES = {
    "Hong Kong S.A.R., China": "Hong Kong",
    "Hong Kong S.A.R. of China": "Hong Kong",
    "Somaliland region": "Somaliland",
    "Somaliland Region": "Somaliland",
    "Taiwan Province of China": "Taiwan",
    "Trinidad and Tobago": "Trinidad & Tobago",
    "North Macedonia": "Macedonia",
    "Northern Cyprus": "North Cyprus",
}

# Explicit country → UN geoscheme region mapping
COUNTRY_TO_REGION = {
    "Afghanistan": "Southern Asia", "Albania": "Southern Europe", "Algeria": "Northern Africa",
    "Angola": "Middle Africa", "Argentina": "South America", "Armenia": "Western Asia",
    "Australia": "Australia and New Zealand", "Austria": "Western Europe", "Azerbaijan": "Western Asia",
    "Bahrain": "Western Asia", "Bangladesh": "Southern Asia", "Belarus": "Eastern Europe",
    "Belgium": "Western Europe", "Belize": "Central America", "Benin": "Western Africa",
    "Bhutan": "Southern Asia", "Bolivia": "South America", "Bosnia and Herzegovina": "Southern Europe",
    "Botswana": "Southern Africa", "Brazil": "South America", "Bulgaria": "Eastern Europe",
    "Burkina Faso": "Western Africa", "Burundi": "Eastern Africa", "Cambodia": "South-Eastern Asia",
    "Cameroon": "Middle Africa", "Canada": "Northern America", "Central African Republic": "Middle Africa",
    "Chad": "Middle Africa", "Chile": "South America", "China": "Eastern Asia",
    "Colombia": "South America", "Comoros": "Eastern Africa", "Congo (Brazzaville)": "Middle Africa",
    "Congo (Kinshasa)": "Middle Africa", "Costa Rica": "Central America", "Croatia": "Southern Europe",
    "Cyprus": "Western Asia", "Czech Republic": "Eastern Europe", "Denmark": "Northern Europe",
    "Djibouti": "Eastern Africa", "Dominican Republic": "Caribbean", "Ecuador": "South America",
    "Egypt": "Northern Africa", "El Salvador": "Central America", "Estonia": "Northern Europe",
    "Ethiopia": "Eastern Africa", "Finland": "Northern Europe", "France": "Western Europe",
    "Gabon": "Middle Africa", "Gambia": "Western Africa", "Georgia": "Western Asia",
    "Germany": "Western Europe", "Ghana": "Western Africa", "Greece": "Southern Europe",
    "Guatemala": "Central America", "Guinea": "Western Africa", "Haiti": "Caribbean",
    "Honduras": "Central America", "Hong Kong": "Eastern Asia", "Hungary": "Eastern Europe",
    "Iceland": "Northern Europe", "India": "Southern Asia", "Indonesia": "South-Eastern Asia",
    "Iran": "Southern Asia", "Iraq": "Western Asia", "Ireland": "Northern Europe",
    "Israel": "Western Asia", "Italy": "Southern Europe", "Ivory Coast": "Western Africa",
    "Jamaica": "Caribbean", "Japan": "Eastern Asia", "Jordan": "Western Asia",
    "Kazakhstan": "Central Asia", "Kenya": "Eastern Africa", "Kosovo": "Southern Europe",
    "Kuwait": "Western Asia", "Kyrgyzstan": "Central Asia", "Laos": "South-Eastern Asia",
    "Latvia": "Northern Europe", "Lebanon": "Western Asia", "Lesotho": "Southern Africa",
    "Liberia": "Western Africa", "Libya": "Northern Africa", "Lithuania": "Northern Europe",
    "Luxembourg": "Western Europe", "Macedonia": "Southern Europe", "Madagascar": "Eastern Africa",
    "Malawi": "Eastern Africa", "Malaysia": "South-Eastern Asia", "Maldives": "Southern Asia",
    "Mali": "Western Africa", "Malta": "Southern Europe", "Mauritania": "Western Africa",
    "Mauritius": "Eastern Africa", "Mexico": "Central America", "Moldova": "Eastern Europe",
    "Mongolia": "Eastern Asia", "Montenegro": "Southern Europe", "Morocco": "Northern Africa",
    "Mozambique": "Eastern Africa", "Myanmar": "South-Eastern Asia", "Namibia": "Southern Africa",
    "Nepal": "Southern Asia", "Netherlands": "Western Europe", "New Zealand": "Australia and New Zealand",
    "Nicaragua": "Central America", "Niger": "Western Africa", "Nigeria": "Western Africa",
    "North Cyprus": "Western Asia", "Norway": "Northern Europe", "Oman": "Western Asia",
    "Pakistan": "Southern Asia", "Palestinian Territories": "Western Asia", "Panama": "Central America",
    "Paraguay": "South America", "Peru": "South America", "Philippines": "South-Eastern Asia",
    "Poland": "Eastern Europe", "Portugal": "Southern Europe", "Puerto Rico": "Caribbean",
    "Qatar": "Western Asia", "Romania": "Eastern Europe", "Russia": "Eastern Europe",
    "Rwanda": "Eastern Africa", "Saudi Arabia": "Western Asia", "Senegal": "Western Africa",
    "Serbia": "Southern Europe", "Sierra Leone": "Western Africa", "Singapore": "South-Eastern Asia",
    "Slovakia": "Eastern Europe", "Slovenia": "Southern Europe", "Somalia": "Eastern Africa",
    "Somaliland": "Eastern Africa", "South Africa": "Southern Africa", "South Korea": "Eastern Asia",
    "South Sudan": "Eastern Africa", "Spain": "Southern Europe", "Sri Lanka": "Southern Asia",
    "Sudan": "Northern Africa", "Suriname": "South America", "Swaziland": "Southern Africa",
    "Sweden": "Northern Europe", "Switzerland": "Western Europe", "Syria": "Western Asia",
    "Taiwan": "Eastern Asia", "Tajikistan": "Central Asia", "Tanzania": "Eastern Africa",
    "Thailand": "South-Eastern Asia", "Togo": "Western Africa", "Trinidad & Tobago": "Caribbean",
    "Tunisia": "Northern Africa", "Turkey": "Western Asia", "Turkmenistan": "Central Asia",
    "Uganda": "Eastern Africa", "Ukraine": "Eastern Europe", "United Arab Emirates": "Western Asia",
    "United Kingdom": "Northern Europe", "United States": "Northern America", "Uruguay": "South America",
    "Uzbekistan": "Central Asia", "Venezuela": "South America", "Vietnam": "South-Eastern Asia",
    "Yemen": "Western Asia", "Zambia": "Eastern Africa", "Zimbabwe": "Eastern Africa",
}


def normalize_country(name):
    """Normalize country name using aliases."""
    return COUNTRY_ALIASES.get(name, name)

standardized_dfs = {}
for year, df in dfs.items():
    df = standardize_columns(df) # Standardize column names first
    df['country'] = df['country'].apply(normalize_country) # Normalize country names
    df['region'] = df['country'].map(COUNTRY_TO_REGION) # Add region
    standardized_dfs[year] = df


## Remove unwanted columns

In [6]:
desired_columns_base = [
    'country',
    'region',
    'happiness_rank',
    'happiness_score',
    'gdp_per_capita',
    'social_support',
    'healthy_life_expectancy',
    'freedom',
    'corruption_perception',
    'generosity'
]
for year in range(2015, 2020):
  standardized_dfs[f"{year}"] = standardized_dfs[f"{year}"][desired_columns_base]

##Adding year suffix for features and make them to column

In [7]:
# Define columns to exclude from year suffixing
EXCLUDE_COLS_FROM_SUFFIX = ['country', 'region']

data_2015 = standardized_dfs["2015"].rename(columns={col: col + " 2015" for col in standardized_dfs["2015"].columns if col not in EXCLUDE_COLS_FROM_SUFFIX})
data_2016 = standardized_dfs["2016"].rename(columns={col: col + " 2016" for col in standardized_dfs["2016"].columns if col not in EXCLUDE_COLS_FROM_SUFFIX})
data_2017 = standardized_dfs["2017"].rename(columns={col: col + " 2017" for col in standardized_dfs["2017"].columns if col not in EXCLUDE_COLS_FROM_SUFFIX})
data_2018 = standardized_dfs["2018"].rename(columns={col: col + " 2018" for col in standardized_dfs["2018"].columns if col not in EXCLUDE_COLS_FROM_SUFFIX})
data_2019 = standardized_dfs["2019"].rename(columns={col: col + " 2019" for col in standardized_dfs["2019"].columns if col not in EXCLUDE_COLS_FROM_SUFFIX})

# Merging Data on Country and Region
complete_data = pd.merge(data_2015, data_2016, on = EXCLUDE_COLS_FROM_SUFFIX, how = "outer")
complete_data = pd.merge(complete_data, data_2017, on = EXCLUDE_COLS_FROM_SUFFIX, how = "outer")
complete_data = pd.merge(complete_data, data_2018, on = EXCLUDE_COLS_FROM_SUFFIX, how = "outer")
complete_data = pd.merge(complete_data, data_2019, on = EXCLUDE_COLS_FROM_SUFFIX, how = "outer")

# Removing data with incomplete values
complete_data = complete_data.dropna()
complete_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 145 entries, 0 to 163
Data columns (total 42 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   country                       145 non-null    object 
 1   region                        145 non-null    object 
 2   happiness_rank 2015           145 non-null    float64
 3   happiness_score 2015          145 non-null    float64
 4   gdp_per_capita 2015           145 non-null    float64
 5   social_support 2015           145 non-null    float64
 6   healthy_life_expectancy 2015  145 non-null    float64
 7   freedom 2015                  145 non-null    float64
 8   corruption_perception 2015    145 non-null    float64
 9   generosity 2015               145 non-null    float64
 10  happiness_rank 2016           145 non-null    float64
 11  happiness_score 2016          145 non-null    float64
 12  gdp_per_capita 2016           145 non-null    float64
 13  social_sup

##Finding Mean and Maximum for One Year

In [8]:
# 2015 Data summary of mean and maximum
numeric_cols_2015 = [col for col in data_2015.columns if col.endswith(' 2015') and pd.api.types.is_numeric_dtype(data_2015[col])]
data_2015_summary = data_2015[numeric_cols_2015].agg(["mean", "max"])
print("2015 Summary:")
print(data_2015_summary)

# 2016 Data summary of mean and maximum
numeric_cols_2016 = [col for col in data_2016.columns if col.endswith(' 2016') and pd.api.types.is_numeric_dtype(data_2016[col])]
data_2016_summary = data_2016[numeric_cols_2016].agg(["mean", "max"])
print("\n2016 Summary:")
print(data_2016_summary)

# 2017 Data summary of mean and maximum
numeric_cols_2017 = [col for col in data_2017.columns if col.endswith(' 2017') and pd.api.types.is_numeric_dtype(data_2017[col])]
data_2017_summary = data_2017[numeric_cols_2017].agg(["mean", "max"])
print("\n2017 Summary:")
print(data_2017_summary)

# 2018 Data summary of mean and maximum
numeric_cols_2018 = [col for col in data_2018.columns if col.endswith(' 2018') and pd.api.types.is_numeric_dtype(data_2018[col])]
data_2018_summary = data_2018[numeric_cols_2018].agg(["mean", "max"])
print("\n2018 Summary:")
print(data_2018_summary)

# 2019 Data summary of mean and maximum (for the entire dataset)
numeric_cols_2019 = [col for col in data_2019.columns if col.endswith(' 2019') and pd.api.types.is_numeric_dtype(data_2019[col])]
data_2019_summary = data_2019[numeric_cols_2019].agg(["mean", "max"])
print("\n2019 Summary:")
print(data_2019_summary)

2015 Summary:
      happiness_rank 2015  happiness_score 2015  gdp_per_capita 2015  \
mean            79.493671              5.375734             0.846137   
max            158.000000              7.587000             1.690420   

      social_support 2015  healthy_life_expectancy 2015  freedom 2015  \
mean             0.991046                      0.630259      0.428615   
max              1.402230                      1.025250      0.669730   

      corruption_perception 2015  generosity 2015  
mean                    0.143422         0.237296  
max                     0.551910         0.795880  

2016 Summary:
      happiness_rank 2016  happiness_score 2016  gdp_per_capita 2016  \
mean            78.980892              5.382185              0.95388   
max            157.000000              7.526000              1.82427   

      social_support 2016  healthy_life_expectancy 2016  freedom 2016  \
mean             0.793621                      0.557619      0.370994   
max            

In [13]:
complete_data.to_csv('complete_happiness_data.csv', index=False)